In [1]:
import httpx, time

API_KEY = "cure-secret-master-key"  # Sesuaikan dengan API_KEY di .env Anda
BASE_URL = "http://127.0.0.1:8005"

sentences = [
    "Bagaimana cara mereset password akun saya?",
    "Saya lupa kata sandi, tolong bantu ganti password.",
    "Harga paket langganan bulanan CRM mulai dari Rp 500.000.",
    "Cara integrasi WhatsApp Business API dengan sistem CRM.",
    "Laporan penjualan bulanan dapat diunduh dalam format Excel dan PDF.",
] * 4  # Total 20 kalimat


async def benchmark_endpoint(client: httpx.AsyncClient, device: str):
    print(f"\n--- Hit Endpoint API (/v1/embeddings) dengan device='{device}' ---")

    # 1. Warmup call (agar jika model CPU baru di-load pertama kali tidak dihitung ke kecepatan murni)
    await client.post(
        "/v1/embeddings",
        json={"input": "warmup", "model": "bge-m3", "device": device},
    )

    # 2. Benchmark call sebenarnya
    t_start = time.perf_counter()
    resp = await client.post(
        "/v1/embeddings",
        json={
            "input": sentences,
            "model": "bge-m3",
            "device": device,
        },
    )
    http_total_ms = (time.perf_counter() - t_start) * 1000
    resp.raise_for_status()
    result = resp.json()

    server_ms = result["duration_ms"]
    print(f"Device Aktif di Server : {result['device'].upper()}")
    print(f"Durasi Murni di Server : {server_ms:.2f} ms ({server_ms / 1000:.4f} detik)")
    print(f"Total Durasi HTTP      : {http_total_ms:.2f} ms ({http_total_ms / 1000:.4f} detik)")
    print(f"Rata-rata per kalimat  : {server_ms / len(sentences):.2f} ms/kalimat")
    return server_ms, http_total_ms


async with httpx.AsyncClient(
    base_url=BASE_URL,
    headers={"X-API-Key": API_KEY},
    timeout=httpx.Timeout(60.0, connect=5.0),
) as client:
    cpu_server_ms, cpu_http_ms = await benchmark_endpoint(client, "cpu")
    gpu_server_ms, gpu_http_ms = await benchmark_endpoint(client, "cuda")

    speedup = cpu_server_ms / gpu_server_ms
    print("\n" + "=" * 55)
    print(f"HASIL ENDPOINT: GPU {speedup:.2f}x LEBIH CEPAT DARI CPU!")
    print(f"Server Inference -> CPU: {cpu_server_ms:.2f} ms | GPU: {gpu_server_ms:.2f} ms")
    print(f"Total HTTP       -> CPU: {cpu_http_ms:.2f} ms | GPU: {gpu_http_ms:.2f} ms")
    print("=" * 55)


--- Hit Endpoint API (/v1/embeddings) dengan device='cpu' ---
Device Aktif di Server : CPU
Durasi Murni di Server : 376.66 ms (0.3767 detik)
Total Durasi HTTP      : 380.86 ms (0.3809 detik)
Rata-rata per kalimat  : 18.83 ms/kalimat

--- Hit Endpoint API (/v1/embeddings) dengan device='cuda' ---
Device Aktif di Server : CUDA
Durasi Murni di Server : 120.03 ms (0.1200 detik)
Total Durasi HTTP      : 124.70 ms (0.1247 detik)
Rata-rata per kalimat  : 6.00 ms/kalimat

HASIL ENDPOINT: GPU 3.14x LEBIH CEPAT DARI CPU!
Server Inference -> CPU: 376.66 ms | GPU: 120.03 ms
Total HTTP       -> CPU: 380.86 ms | GPU: 124.70 ms


In [5]:
import httpx, numpy as np, time
from pathlib import Path
from pypdf import PdfReader

API_KEY = "cure-secret-master-key"  # Sesuaikan dengan API_KEY di .env
BASE_URL = "http://127.0.0.1:8005"
filename = "data/Employee Knowledge warna warni media sby.pdf"

# 1. Cari path file PDF (cek di folder saat ini atau naik 1 level ke root project)
pdf_path = Path(filename)
if not pdf_path.exists():
    pdf_path = Path("..") / filename

print(f"Membaca file PDF: {pdf_path.resolve()}")
reader = PdfReader(str(pdf_path))
num_pages = len(reader.pages)

# 2. Ekstrak teks dan potong menjadi chunks (Ukuran 500 karakter, overlap 100 karakter)
full_text = "\n".join([page.extract_text() or "" for page in reader.pages])
chunk_size = 500
overlap = 100

chunks = [
    full_text[i : i + chunk_size].strip()
    for i in range(0, len(full_text), chunk_size - overlap)
    if len(full_text[i : i + chunk_size].strip()) > 30
]

print(f"Total Halaman PDF : {num_pages} halaman")
print(f"Total Karakter    : {len(full_text):,} karakter")
print(f"Total Chunks RAG  : {len(chunks)} chunks siap di-embed")


# 3. Fungsi Benchmark Endpoint untuk Seluruh Chunks Dokumen
async def benchmark_pdf_endpoint(client: httpx.AsyncClient, device: str):
    print(f"\n--- Memproses {len(chunks)} Chunks PDF di {device.upper()} ---")

    # Warmup 1 teks
    await client.post(
        "/v1/embeddings",
        json={"input": "warmup", "model": "bge-m3", "device": device},
    )

    t_start = time.perf_counter()
    resp = await client.post(
        "/v1/embeddings",
        json={
            "input": chunks,
            "model": "bge-m3",
            "device": device,
        },
    )
    http_total_ms = (time.perf_counter() - t_start) * 1000
    resp.raise_for_status()
    result = resp.json()

    server_ms = result["duration_ms"]
    throughput = len(chunks) / (server_ms / 1000)

    print(f"Device Aktif           : {result['device'].upper()}")
    print(f"Durasi Server Inference: {server_ms:.2f} ms ({server_ms / 1000:.2f} detik)")
    print(f"Total Durasi HTTP      : {http_total_ms:.2f} ms ({http_total_ms / 1000:.2f} detik)")
    print(f"Throughput Kecepatan   : {throughput:.1f} chunks/detik")
    return server_ms, http_total_ms, result["data"]


# 4. Jalankan Benchmark CPU vs GPU
async with httpx.AsyncClient(
    base_url=BASE_URL,
    headers={"X-API-Key": API_KEY},
    timeout=httpx.Timeout(300.0, connect=10.0),  # Timeout 5 menit karena CPU bisa lama untuk PDF tebal
) as client:
    cpu_ms, cpu_http_ms, _ = await benchmark_pdf_endpoint(client, "cpu")
    gpu_ms, gpu_http_ms, gpu_data = await benchmark_pdf_endpoint(client, "cuda")

    speedup = cpu_ms / gpu_ms
    print("\n" + "=" * 60)
    print(f"HASIL DOKUMEN PDF ({len(chunks)} Chunks):")
    print(f"GPU RTX 3050 {speedup:.2f}x LEBIH CEPAT DARI CPU!")
    print(f"Waktu CPU : {cpu_ms / 1000:.2f} detik  vs  Waktu GPU : {gpu_ms / 1000:.2f} detik")
    print("=" * 60)

    # 5. BONUS: Tes Pencarian Semantik (RAG Search) ke Dokumen PDF
    query = "Apa isi pokok atau informasi perusahaan PT Warna Warni Media?"
    q_resp = await client.post(
        "/v1/embeddings",
        json={"input": query, "model": "bge-m3", "device": "cuda"},
    )
    query_vec = np.array(q_resp.json()["data"][0]["embedding"])
    doc_vecs = np.array([item["embedding"] for item in gpu_data])

    scores = doc_vecs @ query_vec
    top_idx = int(np.argmax(scores))
    print(f"\n[Tes Search Query]: '{query}'")
    print(f"[Skor Kemiripan  ]: {scores[top_idx]:.4f}")
    print(f"[Chunk Terbaik #{top_idx}]:\n{chunks[top_idx][:300]}...")

Membaca file PDF: C:\Users\aiai\Documents\Development\Projects\fastapi-api-source\tests\data\Employee Knowledge warna warni media sby.pdf
Total Halaman PDF : 26 halaman
Total Karakter    : 6,977 karakter
Total Chunks RAG  : 18 chunks siap di-embed

--- Memproses 18 Chunks PDF di CPU ---
Device Aktif           : CPU
Durasi Server Inference: 2226.08 ms (2.23 detik)
Total Durasi HTTP      : 2230.52 ms (2.23 detik)
Throughput Kecepatan   : 8.1 chunks/detik

--- Memproses 18 Chunks PDF di CUDA ---
Device Aktif           : CUDA
Durasi Server Inference: 265.94 ms (0.27 detik)
Total Durasi HTTP      : 270.49 ms (0.27 detik)
Throughput Kecepatan   : 67.7 chunks/detik

HASIL DOKUMEN PDF (18 Chunks):
GPU RTX 3050 8.37x LEBIH CEPAT DARI CPU!
Waktu CPU : 2.23 detik  vs  Waktu GPU : 0.27 detik

[Tes Search Query]: 'Apa isi pokok atau informasi perusahaan PT Warna Warni Media?'
[Skor Kemiripan  ]: 0.6272
[Chunk Terbaik #0]:
Handout Onboarding System
PT WARNA WARNI MEDIA SURABAYA
EMPLOYEE KNOWLEDGE
•D